# Tutorial 5: Working with Grid Plates

So far we have worked with `GridImage` objects without paying much attention
to the *grid* part. In this tutorial you will learn what makes a `GridImage`
special — it knows the row-and-column layout of your plate, so you can
inspect individual wells, count colonies per grid section, and visualize
the grid structure.

**What you will learn:**

1. How `GridImage` differs from `Image`
2. Access grid properties (rows, columns)
3. Run detection, then query per-colony grid assignments
4. Extract a single well as a subimage
5. Count colonies per grid section
6. Visualize the grid overlay

## Imports

In [ ]:
import phenotypic as pht
from phenotypic.data import load_yeast_plate
from phenotypic.enhance import GaussianBlur, CLAHE
from phenotypic.detect import OtsuDetector

## Load the Plate

`load_yeast_plate()` returns a `GridImage` — an `Image` that also knows
about the grid layout. Our *Rhodotorula* plate is arranged in a standard
96-well format: 8 rows by 12 columns.

In [ ]:
plate = load_yeast_plate()
print(f"Type:    {type(plate).__name__}")
print(f"Rows:    {plate.grid.nrows}")
print(f"Columns: {plate.grid.ncols}")

## Detect Colonies First

Grid features like colony assignments and section counts require detected
objects. Let's run the same enhance-and-detect pipeline from earlier tutorials.

In [ ]:
pipeline = pht.ImagePipeline(
    ops=[GaussianBlur(sigma=2.0), CLAHE(clip_limit=0.01), OtsuDetector()]
)
plate = pipeline.apply(plate)

## Visualize the Grid Overlay

The overlay view becomes especially powerful on grid plates — you can see
the detected colonies *and* the grid boundaries at the same time.

In [ ]:
plate.dash(overlay=True, show_grid=True)

The dashed lines show the grid boundaries. Each rectangular region is one
grid section (corresponding to one well on the physical plate).

## Query Grid Assignments

The `.grid.info()` method returns a DataFrame with one row per detected
colony, including its grid position — which row, which column, and which
flattened section number it belongs to.

In [ ]:
info = plate.grid.info()
info.head(10)

Key columns:

- **RowNum** / **ColNum** — grid row and column (0-indexed)
- **SectionNum** — flattened section index (0 to nrows × ncols − 1)
- **CenterRR** / **CenterCC** — colony centroid in pixel coordinates
- **MinRR**, **MaxRR**, **MinCC**, **MaxCC** — bounding box

## Extract a Single Well

You can pull out any grid section as a standalone subimage using bracket
indexing on the `.grid` accessor. Let's look at the well in row 0, column 0
(top-left corner).

In [ ]:
well = plate.grid[0, 0]
well.dash()

You can also extract an entire row or column with slicing:

```python
first_row = plate.grid[0, :]     # All 12 wells in row 0
third_col = plate.grid[:, 2]     # All 8 wells in column 2
```

## Count Colonies per Grid Section

How many colonies are in each well? `.grid.get_section_counts()` gives you
a quick summary.

In [ ]:
counts = plate.grid.get_section_counts()
counts.head(10)

The index is the section number and the value is the colony count. Sections
with zero colonies are omitted by default.

## Summary

You now know how to work with grid plates in PhenoTypic:

- **`plate.grid.nrows`** / **`.ncols`** — grid dimensions
- **`plate.grid.info()`** — per-colony DataFrame with grid row, column, and section
- **`plate.grid[row, col]`** — extract a single well as a subimage
- **`plate.grid.get_section_counts()`** — colony counts per section
- **`plate.dash(overlay=True, show_grid=True)`** — visual grid overlay

Grid awareness is what makes PhenoTypic especially powerful for arrayed
colony assays — every colony knows which well it belongs to.

**Next up:** [Tutorial 6: Batch Processing](06_batch_processing.ipynb) —
process many plates at once using the command-line interface.